# v25 固定ラグ系統スムーザー Kaggle 提出ノート(自己完結 / PF=r0固定)

**内容**: test の各評価行(TVT_input=NaN)に対し、
**単一 r0(r0_seed32) の前向きPF + 固定ラグ系統スムーザー(先読みL)→ forwardとblend** → submission.csv。

- FFBS版(submit_ffbsi_v25.ipynb)の**固定ラグ版**。エンジンは tune_smoother_r0_v25.py と同一。
- PF=r0固定。カテゴリ分岐/scale/beam/hold は入れない(平滑が主役)。
- **SMOOTH_LAG**=先読みステップ数 L(tuner smoother_r0.json の best)。**SMOOTH_BLEND**=(1-b)*fwd+b*smoothed。
- バッチ処理(_pad)なので FFBS(坑井ごと)より速い。メモリはバッファ B*L*N(=CHUNK*N_SEED*L*N)に注意。
- **CLAMP_FT**(任意): smoothed が forward から±これ超で外れた行は forward に戻す(既定 None=無効。固定ラグは通常安定)。
- **★beam + hold(カテゴリ別)** 搭載(FFBS版と同一。診断で FFBS上でも効いたため。固定ラグでも同様に載せる)。
- **APPLY_PP**(任意): tauランプ+sg。
- test は隠しセットに置換され rerun。**test 実在列(MD,X,Y,Z,GR,TVT_input + typewell)だけ**から生成。
- 注意: 固定ラグと FFBS は**別手法**(近似トレードオフが違う)。L 過大で系譜退化しうる。これは別軸提出。


In [ ]:
# ===== import =====
import os, glob, json, warnings, time
import numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.signal import savgol_filter
from numba import njit
warnings.filterwarnings("ignore"); np.seterr(all="ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32
print("device:", DEVICE, "| torch:", torch.__version__)

In [ ]:
# ===== データパス自動判定 =====
CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    "C:/Users/kosaka256/Documents/rogii_claude/rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TEST = COMP / "test"
print("COMP:", COMP, "| test exists:", TEST.exists())

In [ ]:
# ===== 設定(tune_smoother_r0_v25.py と揃える) =====
SEED        = 0
N_SEED      = 128          # r0_seed32 regime
SMOOTH_LAG  = 192         # 先読みステップ数 L(smoother_r0.json の best。実測 lag192で train -0.229)
SMOOTH_BLEND = 1.0        # 最終=(1-b)*forward + b*smoothed。OOFは ~0.94 を示唆
CHUNK       = 80          # GPUバッチ坑井数。バッファ B*L*N(=CHUNK*N_SEED*L*N)。OOMなら下げる
CLAMP_FT    = None        # smoothedがforwardから±これ超で外れた行はforwardに戻す。None=無効

# --- カテゴリ別 beam + hold(tuned-heuristic-only-cat6.ipynb と同一値) ---
USE_BEAM = True
USE_HOLD = True
N_EVAL_THRESHOLD  = 4840.0
Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
BEAM_W = {0: 0.0, 1: 0.0, 2: 0.2, 3: 0.0, 4: 0.05, 5: 0.2}
HOLD_W = {0: 0.2, 1: 0.15, 2: 0.15, 3: 0.15, 4: 0.05, 5: 0.05}
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1), (10, 14.0, 90.0, 5),
    (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3), (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2),
    (15, 10.0, 80.0, 4), (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0),
]

APPLY_PP    = True        # tauランプ+sg。単体を見たいなら False
PP_PARAMS   = {"alpha": 1.0, "tau": 85}
SG_W, SG_P  = 17, 3

# 固定 PF = r0_seed32 (use_phys=false)
R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}
print(f"PF=r0_seed32 n_seed={N_SEED} smooth_lag={SMOOTH_LAG} blend={SMOOTH_BLEND} chunk={CHUNK} "
      f"clamp={CLAMP_FT} beam={USE_BEAM} hold={USE_HOLD} pp={APPLY_PP}")

In [ ]:
# ===== エンジン(tune_smoother_r0_v25.py と同一・インライン) =====
def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def well_code(c):
    n_bin = int(c["n_eval"] > N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(Z_SPAN_THRESHOLDS, c["z_span"], side="right"))
    return n_bin + 2*z_bin

def build_test(wid):
    hw = pd.read_csv(TEST / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TEST / f"{wid}__typewell.csv").sort_values("TVT")
    if "TVT_input" not in hw.columns or "TVT" not in tw.columns:
        return None
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0 or len(kn) < 1:
        return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    if len(tw_tvt) < 3:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    z_ev = ev["Z"].to_numpy(float)
    n_eval = int(len(z_ev)); z_span = float(np.nanmax(z_ev)-np.nanmin(z_ev)) if len(z_ev) else 0.0
    ev_ids = [f"{wid}_{i}" for i in ev.index]
    return dict(md=ev["MD"].to_numpy(float), z=z_ev, gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos,
                kn_tvtin=kn["TVT_input"].to_numpy(float), tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb,
                ls=ls, ir=ir, ev_ids=ev_ids, n_eval=n_eval, z_span=z_span,
                last_known_md=float(kn["MD"].iloc[-1]), last_known_tvt=float(kn["TVT_input"].iloc[-1]))

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf)
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs*P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"]+1e-6)))
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]))


def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik


def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s


# ===== beam search(7.191移植, numba) =====
@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni])**2/es + mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

def _nn(arr, val): return int(np.abs(arr - val).argmin())

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, int(bs), float(mc), float(es))
    return tw_tvt[path].astype(np.float32)

def run_beam(c):
    tw_tvt = c["tw_tvt"].astype(np.float64)
    fb = float(np.nanmean(c["tw_gr"])) if np.isfinite(np.nanmean(c["tw_gr"])) else 0.0
    tw_gr = np.where(np.isfinite(c["tw_gr"]), c["tw_gr"], fb).astype(np.float64)
    hgr = c["gr_filled"][c["ev_pos"]]; last_tvt = c["last_known_tvt"]
    res = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r) in BEAM_CONFIGS]
    return np.stack(res, 0).mean(0).astype(np.float64)


def post_process(pred, c):
    pred = np.asarray(pred, dtype=np.float64)
    if not APPLY_PP:
        return pred
    md_since = np.maximum(c["md"] - c["last_known_md"], 0.0)
    d = (pred - c["last_known_tvt"]) * (1.0 - np.exp(-md_since / float(PP_PARAMS["tau"]))) * float(PP_PARAMS.get("alpha", 1.0))
    out = c["last_known_tvt"] + d
    n = len(out); wl = min(SG_W, n)
    if wl % 2 == 0: wl -= 1
    return savgol_filter(out, wl, SG_P) if wl >= SG_P + 2 else out

In [ ]:
# ===== XY近傍(test←train) + 疑似typewell(mixed grid) + 距離連動α =====
# 議論した v27 XY近傍版を提出に注入。PFパラメータ(R0/N_SEED/SMOOTH_LAG…)は上のまま。
K_MAX      = 5          # 近傍の最大本数
MAX_DIST   = 1500.0     # これより遠いtrain坑井は参照にしない(α cutoff と整合。far参照ノイズ除去)
MIX_W      = 1.0        # 疑似typewell混合比(covered帯; 1.0=実HWテンプレで完全置換)
NEED_REFS  = 2          # 近傍がこの本数未満なら混ぜない(=現行tw)。2本ないと安定しない実測
SPLIT      = "test"     # "test"=提出 / "train"=train坑井でOOF検証(末尾の採点セルで真値TVTと比較)
N_WELLS    = None       # 検証を軽くする時に坑井数制限(例50)。None=全部
TRAIN = COMP / "train"
TEST  = COMP / SPLIT    # build_test/test_neighbors はこの TEST を読む(train検証時はtrainを読む)

# α(nn_dist): <=250→0.7 / 500→0.6 / 1000→0.45 / 1500→0.3 / >1500→0(混ぜない=先端)
def alpha_dist(d):
    if d <= 250: return 0.7
    if d > 1500: return 0.0
    return float(np.interp(d, [250, 500, 1000, 1500], [0.7, 0.6, 0.45, 0.3]))

# train坑井の XY重心
_train_wids = sorted(os.path.basename(p).split("__")[0]
                     for p in glob.glob(str(TRAIN / "*__horizontal_well.csv")))
_tw_cent = {}
for _w in _train_wids:
    _h = pd.read_csv(TRAIN / f"{_w}__horizontal_well.csv", usecols=["X", "Y"])
    _tw_cent[_w] = (float(_h["X"].mean()), float(_h["Y"].mean()))
_tw_xy = np.array([_tw_cent[w] for w in _train_wids])
print("train坑井(参照プール):", len(_train_wids))

# 参照train坑井の (GR, TVT) キャッシュ
_REFCACHE = {}
def ref_gr_tvt(wid):
    if wid not in _REFCACHE:
        _h = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv", usecols=["GR", "TVT"])
        _REFCACHE[wid] = (_h["GR"].to_numpy(float), _h["TVT"].to_numpy(float))
    return _REFCACHE[wid]

# test坑井の XY近傍(同名train=可視testのサブセットは除外=リーク防止)
def test_neighbors(test_wid):
    _h = pd.read_csv(TEST / f"{test_wid}__horizontal_well.csv", usecols=["X", "Y"])
    cx, cy = float(_h["X"].mean()), float(_h["Y"].mean())
    d = np.hypot(_tw_xy[:, 0] - cx, _tw_xy[:, 1] - cy)
    out = []
    for j in np.argsort(d):
        w = _train_wids[j]
        if w == test_wid: continue
        if d[j] > MAX_DIST: break
        out.append((w, float(d[j])))
        if len(out) >= K_MAX: break
    return out

# 疑似typewell(参照HW GRのTVTビン median)を typewell grid に混ぜた make_inputs
def make_inputs_mixed(c, refs, P, w_hw=MIX_W):
    base = make_inputs(c, P)
    gmin = base["gmin"]; step = base["gst"]; G = len(base["gg"])
    rt = np.concatenate([r[1] for r in refs])
    rg = np.concatenate([_smooth(r[0], float(np.nanmean(r[0])), P["hgr_smooth_r"]).astype(float) for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt, rg = rt[m], rg[m]
    idx = np.round((rt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, rgv = idx[ok], rg[ok]
    ghw = np.full(G, np.nan)
    med = pd.Series(rgv).groupby(idx).median(); ghw[med.index.to_numpy()] = med.to_numpy()  # median pool
    have = np.isfinite(ghw)
    gg = base["gg"].copy(); gg[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    base = dict(base); base["gg"] = gg
    return base
print(f"XY近傍注入 ready | K_MAX={K_MAX} MAX_DIST={MAX_DIST:.0f} MIX_W={MIX_W} NEED_REFS={NEED_REFS} α=距離連動")

In [ ]:
# ===== test 推論: tw-PF / mix-PF 2パス → 距離連動αで位置ブレンド → beam+hold+pp(最後に1回) =====
wids = sorted(set(Path(p).stem.replace("__horizontal_well", "")
                  for p in glob.glob(str(TEST / "*__horizontal_well.csv"))))
print("test wells:", len(wids))
CACHE = {}
for wid in wids:
    try:
        c = build_test(wid)
    except Exception:
        c = None
    if c is not None: CACHE[wid] = c
usable = [w for w in wids if w in CACHE]
if N_WELLS: usable = usable[:N_WELLS]
print("usable wells:", len(usable), "| SPLIT:", SPLIT)

# 近傍とα(test坑井ごと)
NB = {w: test_neighbors(w) for w in usable}
def alpha_of(w):
    nb = NB[w]
    return alpha_dist(nb[0][1]) if len(nb) >= NEED_REFS else 0.0
mix_wids = [w for w in usable if alpha_of(w) > 0]
print(f"混合対象(α>0, 近傍>={NEED_REFS}本 & nn<=1500m): {len(mix_wids)} / {len(usable)} 坑井")

P = {**R0, "smooth_lag": SMOOTH_LAG}
def _blend_fs(fwd, sm):
    if CLAMP_FT is not None:
        bad = np.abs(sm - fwd) > CLAMP_FT
        if bad.any(): sm = np.where(bad, fwd, sm)
    return (1.0 - SMOOTH_BLEND) * fwd + SMOOTH_BLEND * sm

# --- パス1: tw-PF(全坑井, typewell grid) ---
OUT_TW = {}; t0 = time.time()
for c0 in range(0, len(usable), CHUNK):
    sub = usable[c0:c0+CHUNK]
    out_f, out_s = run_smoother_ens([make_inputs(CACHE[w], P) for w in sub], P, DEVICE, SEED, N_SEED)
    for w, fwd, sm in zip(sub, out_f, out_s): OUT_TW[w] = _blend_fs(fwd, sm)
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  tw {min(c0+CHUNK,len(usable))}/{len(usable)} ({time.time()-t0:.0f}s)")

# --- パス2: mix-PF(混合対象のみ, mixed grid=疑似typewell) ---
OUT_MIX = {}
for c0 in range(0, len(mix_wids), CHUNK):
    sub = mix_wids[c0:c0+CHUNK]
    inps = [make_inputs_mixed(CACHE[w], [ref_gr_tvt(r) for r, _ in NB[w]], P) for w in sub]
    out_f, out_s = run_smoother_ens(inps, P, DEVICE, SEED, N_SEED)
    for w, fwd, sm in zip(sub, out_f, out_s): OUT_MIX[w] = _blend_fs(fwd, sm)
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  mix {min(c0+CHUNK,len(mix_wids))}/{len(mix_wids)} ({time.time()-t0:.0f}s)")

# --- 距離連動α で位置ブレンド → beam+hold+pp(最後に1回) ---
rows = []
for w in usable:
    c = CACHE[w]; a = alpha_of(w)
    pos = OUT_TW[w] if a == 0 else (1.0 - a) * OUT_TW[w] + a * OUT_MIX[w]
    code = well_code(c)
    bw = BEAM_W.get(code, 0.0) if USE_BEAM else 0.0
    hwt = HOLD_W.get(code, 0.0) if USE_HOLD else 0.0
    pred = np.asarray(pos, dtype=np.float64).copy()
    if bw > 0:
        beam = run_beam(c); m = min(len(pred), len(beam)); pred[:m] = (1.0 - bw) * pred[:m] + bw * beam[:m]
    if hwt > 0:
        pred = (1.0 - hwt) * pred + hwt * c["last_known_tvt"]
    pred = post_process(pred, c)
    ids = c["ev_ids"]; n = min(len(ids), len(pred))
    rows.extend((ids[i], float(pred[i])) for i in range(n))
pred_df = pd.DataFrame(rows, columns=["id", "tvt"])
print("predicted rows:", len(pred_df), f"({time.time()-t0:.0f}s)")

In [ ]:
# ===== 提出ファイル作成 =====
ss = pd.read_csv(COMP / "sample_submission.csv")
sub = ss[["id"]].merge(pred_df, on="id", how="left")
fill = float(np.nanmedian(pred_df["tvt"])) if len(pred_df) else 0.0
n_missing = int(sub["tvt"].isna().sum())
sub["tvt"] = sub["tvt"].fillna(fill)
sub.to_csv("submission.csv", index=False)
print("submission:", sub.shape, "| filled(missing):", n_missing, "| fill_value:", round(fill, 3))
sub.head()

In [ ]:
# ===== (SPLIT=="train"のみ) 真値TVTで OOF 採点: XY混合版 vs tw版(共に beam+hold+pp) =====
if SPLIT == "train":
    def _finalize(pos, c):
        code = well_code(c)
        bw = BEAM_W.get(code, 0.0) if USE_BEAM else 0.0
        hwt = HOLD_W.get(code, 0.0) if USE_HOLD else 0.0
        pred = np.asarray(pos, dtype=np.float64).copy()
        if bw > 0:
            beam = run_beam(c); m = min(len(pred), len(beam)); pred[:m] = (1.0 - bw) * pred[:m] + bw * beam[:m]
        if hwt > 0:
            pred = (1.0 - hwt) * pred + hwt * c["last_known_tvt"]
        return post_process(pred, c)
    em = []; et = []
    for w in usable:
        c = CACHE[w]; a = alpha_of(w)
        hw = pd.read_csv(TRAIN / f"{w}__horizontal_well.csv")
        ev = hw["TVT_input"].isna().to_numpy(); true = hw["TVT"].to_numpy(float)[ev]
        pm = _finalize(OUT_TW[w] if a == 0 else (1.0 - a) * OUT_TW[w] + a * OUT_MIX[w], c)
        pt = _finalize(OUT_TW[w], c)
        n = min(len(true), len(pm)); em.append(pm[:n] - true[:n]); et.append(pt[:n] - true[:n])
    _r = lambda e: float(np.sqrt(np.mean(e[np.isfinite(e)] ** 2)))
    EM = np.concatenate(em); ET = np.concatenate(et)
    print("=== train OOF overall RMSE (提出と同一: PF→距離α mix→beam+hold+pp) ===")
    print("  tw のみ  : %.4f" % _r(ET))
    print("  XY混合   : %.4f  (改善 %+.4f)" % (_r(EM), _r(ET) - _r(EM)))
    print("  混合坑井 : %d / %d  (MAX_DIST=%.0f, NEED_REFS=%d, α=距離連動)" % (len(mix_wids), len(usable), MAX_DIST, NEED_REFS))
else:
    print("SPLIT=test: 提出モード(採点なし)。train検証は SPLIT='train' に。")